# 685221 Visualization Walkthrough

This notebook is a clean walkthrough of the current `685221` rendering and analysis outputs already present in the surrounding workspace.

It covers:

- sagittal 3D CCF rendering
- aligned SWC reconstructions inside the atlas
- exact-coordinate level-0 XY MIPs
- automated local carveouts and SWC-derived masks
- XY-only PCA and UMAP
- morphology-only PCA and UMAP


In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

cwd = Path.cwd().resolve()
repo_root = cwd.parent if cwd.name == 'notebooks' else cwd
sys.path.insert(0, str(repo_root / 'src'))

from lc_walkthrough import PATHS


def show_image(path, title=None, figsize=(8, 6)):
    path = Path(path)
    img = Image.open(path)
    plt.figure(figsize=figsize)
    plt.imshow(img)
    plt.axis('off')
    if title:
        plt.title(title)
    plt.show()


def show_row(paths, titles=None, figsize=(18, 6)):
    titles = titles or [None] * len(paths)
    fig, axes = plt.subplots(1, len(paths), figsize=figsize)
    if len(paths) == 1:
        axes = [axes]
    for ax, path, title in zip(axes, paths, titles):
        img = Image.open(path)
        ax.imshow(img)
        ax.axis('off')
        if title:
            ax.set_title(title)
    plt.tight_layout()
    plt.show()


def read_json(path):
    with open(path) as f:
        return json.load(f)


PATHS


## 1. Latest CCF Render

The current atlas render used here is the latest clean full-brain sagittal glass render from the workspace.

The underlying renderer is:

- `scripts/render_ccf_sagittal_3d.py`


In [ ]:
print(PATHS['latest_ccf_render'])
print(PATHS['render_script'])
show_image(PATHS['latest_ccf_render'], title='Latest sagittal 3D CCF render', figsize=(12, 7))


## 2. CCF With Reconstructions Inside the Atlas

These are atlas renders with aligned SWC reconstructions overlaid inside the same CCF view. They are useful for quickly validating where a traced reconstruction sits in atlas space before going back to specimen-space image data.


In [ ]:
show_row(
    [PATHS['ccf_overlay_n053'], PATHS['ccf_overlay_n047']],
    titles=['Aligned SWC overlay: N053', 'Aligned SWC overlay: N047'],
    figsize=(16, 6),
)


## 3. Exact-Coordinate Level-0 XY MIPs

The MIPs in this project are generated as paired outputs per coordinate:

- base view: `2 mm x 2 mm`, `z1000`
- zoomed view: `1 mm x 1 mm`, `z500`
- both from level `0`

The generator used here is:

- `scripts/generate_exact_level0_dual_mips.py`


In [ ]:
mip_folder = PATHS['mip_grouped_dir'] / 'N038_x37794p234_y11058p196_z8589p203'
base_png = sorted(mip_folder.glob('*z1000*.png'))[0]
zoom_png = sorted(mip_folder.glob('*zoom2_z500*.png'))[0]

print('MIP folder:', mip_folder)
print('Generator:', PATHS['mip_script'])
show_row([base_png, zoom_png], titles=['Base view: z1000', 'Zoomed view: z500'], figsize=(16, 6))


## 4. Automated Local Carveouts and SWC-Derived Masks

The carveout pipeline uses an assigned raw SWC to define a local mask around a coordinate of interest, then extracts:

- raw volume
- binary mask volume
- masked volume
- QC projections

Current settings used in these examples:

- local SWC mining radius: `250 um`
- padding around selected segment bbox: `32 vox`
- max extent per axis: `512 vox`
- mask dilation radius: `8 vox`

The main generator is:

- `scripts/generate_masked_local_swc_carveouts.py`


In [ ]:
qc_path = PATHS['mask_qc_example']
overview_path = PATHS['mask_overview_xy']
meta_path = qc_path.parent / 'metadata.json'
meta = read_json(meta_path)

print('QC example:', qc_path)
print('Generator:', PATHS['carveout_script'])
print('Metadata keys:', list(meta.keys())[:12])
print('Volume shape (zyx):', meta.get('volume_shape_zyx'))
print('Selected segment count:', meta.get('selected_segment_count'))
print('Mask voxels:', meta.get('mask_voxels'))

show_image(qc_path, title='Detailed carveout mask QC', figsize=(14, 10))
show_image(overview_path, title='Overview of traced masked carveout examples', figsize=(14, 10))


## 5. XY-Only PCA and UMAP

These analyses use only metrics derived from XY views and XY max projections. This is the cleaner choice when the non-XY views are blurrier and likely to dilute the morphologically useful structure.

Inputs live under:

- `685221/analysis/balanced_carveouts_120_combined_xy_only`
- `685221/analysis/balanced_carveouts_120_combined_xy_only_umap`


In [ ]:
xy_pca_summary = read_json(PATHS['xy_pca_dir'] / 'feature_summary.json')
xy_umap_summary = read_json(PATHS['xy_umap_dir'] / 'umap_summary.json')
print('XY-only PCA summary:')
print(json.dumps(xy_pca_summary, indent=2)[:1200])
print('
XY-only UMAP summary:')
print(json.dumps(xy_umap_summary, indent=2))

show_row(
    [
        PATHS['xy_pca_dir'] / 'masked_neurite_xy_only_pca.png',
        PATHS['xy_pca_dir'] / 'masked_neurite_xy_only_pca_3d_regions.png',
    ],
    titles=['XY-only PCA', 'XY-only PCA (3D)'],
    figsize=(16, 6),
)

show_row(
    [
        PATHS['xy_umap_dir'] / 'masked_neurite_xy_only_umap.png',
        PATHS['xy_umap_dir'] / 'masked_neurite_xy_only_umap_3d_regions.png',
    ],
    titles=['XY-only UMAP', 'XY-only UMAP (3D)'],
    figsize=(16, 6),
)


## 6. Morphology-Only PCA and UMAP

This analysis drops most bulk size and mean-brightness terms and focuses more heavily on features tied to local punctateness, beadiness, XY shape, and path-level structure.

Inputs live under:

- `685221/analysis/balanced_carveouts_120_combined_morphology_only`
- `685221/analysis/balanced_carveouts_120_combined_morphology_only_umap`


In [ ]:
morph_pca_summary = read_json(PATHS['morph_pca_dir'] / 'feature_summary.json')
morph_umap_summary = read_json(PATHS['morph_umap_dir'] / 'umap_summary.json')
print('Morphology-only PCA summary:')
print(json.dumps(morph_pca_summary, indent=2)[:1200])
print('
Morphology-only UMAP summary:')
print(json.dumps(morph_umap_summary, indent=2))

show_row(
    [
        PATHS['morph_pca_dir'] / 'masked_neurite_morphology_only_pca.png',
        PATHS['morph_pca_dir'] / 'masked_neurite_morphology_only_pca_3d_regions.png',
    ],
    titles=['Morphology-only PCA', 'Morphology-only PCA (3D)'],
    figsize=(16, 6),
)

show_row(
    [
        PATHS['morph_umap_dir'] / 'masked_neurite_morphology_only_umap.png',
        PATHS['morph_umap_dir'] / 'masked_neurite_morphology_only_umap_3d_regions.png',
    ],
    titles=['Morphology-only UMAP', 'Morphology-only UMAP (3D)'],
    figsize=(16, 6),
)


## 7. Reproducible Script Entry Points

These are the main scripts used in the current workspace for each stage.


In [ ]:
scripts = {
    'CCF render': PATHS['render_script'],
    'Exact-coordinate MIPs': PATHS['mip_script'],
    'Masked carveouts': PATHS['carveout_script'],
    'XY-only PCA/features': PATHS['xy_pca_script'],
    'Morphology-only PCA': PATHS['morph_pca_script'],
    'XY-only UMAP': PATHS['xy_umap_script'],
}
for name, path in scripts.items():
    print(f'{name}: {path}')
